In [ ]:
import sys
sys.path.append('../Model_info')
import os
import Stdinput as orctm
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime,timedelta
import time

In [ ]:
INI=orctm.Orctm_cvcm('ORCTM_cvcm','initial fields for Temperature')
latp_grid = INI.model_info_p('latp_grid')
lonp_grid = INI.model_info_p('lonp_grid')
zzout = INI.model_info_p('zzout')
mask = INI.model_info_p('weto_p')
mask = mask.astype(int)

In [ ]:
cmems_file = 'CMEMS-For-CVCM_INI_file.nc'
ini_file = nc.Dataset(cmems_file,mode='r',format='netCDF4')
#print(ini_file.variables)
latitude = ini_file.variables['latitude'][:]
longitude = ini_file.variables['longitude'][:]
longitude_grid,latitude_grid = np.meshgrid(longitude,latitude)
DEP = ini_file.variables['depth'][:]
so  = ini_file.variables['so'][:,:,:,:]
so  = so.squeeze()
so  = so.filled(fill_value = np.nan)
nc.Dataset.close(ini_file)

In [ ]:
start_time = time.time()
# Start the timer
keo = np.size(so,axis=0)
so_keo = np.zeros((INI.ie_g,INI.je_g,keo))
so_ini = np.zeros((INI.ie_g,INI.je_g,INI.ke))
Lat_grid = INI.rearrange_grid_to_orctm(latitude_grid)
Lon_grid = INI.rearrange_grid_to_orctm(longitude_grid)
for k in range(keo):
    print('Layer: ',DEP[k],' m')
    so_grid  = INI.rearrange_grid_to_orctm(so[k,:,:])
    land = np.zeros_like(so_grid)
    land[np.isnan(so_grid)]=1  # land 1 Sea: 0 !!!!
    if np.sum(land) != np.size(land):
        so_keo[:,:,k] = INI.Grid_bilinear_interpolation( \
            so_grid,land,Lat_grid,Lon_grid,latp_grid,lonp_grid,landvalue=np.nanmean(so_grid))

so_keo[so_keo==0]=np.nan # due to the landmask
so_ini = INI.vertical_interp2d(so_keo,DEP,zzout,mask)
so_ini[mask==0] = np.nan
# check obc
# minimum
so_ini = np.where(so_ini <= np.nanmin(so[:,:,:]),np.nanmin(so[:,:,:]),so_ini)
# maximum
so_ini = np.where(so_ini >= np.nanmax(so[:,:,:]),np.nanmax(so[:,:,:]),so_ini)
# Print elapsed time
end_time = time.time()
print(f"Elapsed time: {end_time - start_time:.6f} seconds")
so_ini[np.isnan(so_ini)==True]=0

In [ ]:
import struct
if1 = 0
if22 = 2
with open('INISAL', 'wb') as inisal:
    for k in range(INI.ke):
        print('Layer: ',int(INI.ZZOUT[k]),' m')
        # Header for INITEM
        header = struct.pack('>qqqq', int(if1), int(if22), int(INI.ZZOUT[k]), int(INI.ie_g * INI.je_g))
        header_len = len(header)
        
        inisal.write(struct.pack(">i", header_len))  # length
        inisal.write(header)                         # record data
        inisal.write(struct.pack(">i", header_len))  # length

        # Data arrays 
        sal_level = so_ini[:,:,k].astype('>f8')  
        sal_level = sal_level.T  
        sal_data = sal_level.tobytes()  
        sal_len = len(sal_data)

        inisal.write(struct.pack(">i", sal_len))  # length
        inisal.write(sal_data)                    # data
        inisal.write(struct.pack(">i", sal_len))  # length
